# City Library Summer Reading Program - Data Project

This notebook combines all the work for Task 1 (Data Gathering and Combination), Task 2 (Data Integrity), and Task 3 (Data Fairness and Version Control).

## Task 1: Data Gathering and Combination

### Part 1 - Five Business Questions
Each question is answered directly against the database using SQL.

In [1]:
"""
Task 1 - Part 1: Five business questions about members and checkouts.
Each answer is printed with a short label so it can be copied into the write-up file.
"""

import sqlite3

DB_PATH = "Library Database"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print("=" * 60)
print("Q1: How much is each member borrowing? (per-member checkout count, including zero)")
print("=" * 60)
query1 = """
SELECT m.member_id, m.first_name, m.last_name, COUNT(c.checkout_id) AS total_checkouts
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY total_checkouts DESC;
"""
cursor.execute(query1)
rows1 = cursor.fetchall()
for row in rows1:
    print(row)
print(f"Total members: {len(rows1)}")

print("\n" + "=" * 60)
print("Q2: Which books match a chosen author pattern?")
print("=" * 60)
# Pattern chosen: authors whose name starts with 'A'
# Change AUTHOR_PATTERN below if you want a different letter/pattern
AUTHOR_PATTERN = "A%"
query2 = """
SELECT book_id, title, author
FROM books
WHERE author LIKE ?;
"""
cursor.execute(query2, (AUTHOR_PATTERN,))
rows2 = cursor.fetchall()
for row in rows2:
    print(row)
print(f"Pattern used: author LIKE '{AUTHOR_PATTERN}'")

print("\n" + "=" * 60)
print("Q3: What are the 5 most popular books (most-borrowed titles)?")
print("=" * 60)
query3 = """
SELECT b.book_id, b.title, b.author, COUNT(c.checkout_id) AS times_borrowed
FROM checkouts c
JOIN books b ON c.book_id = b.book_id
GROUP BY b.book_id, b.title, b.author
ORDER BY times_borrowed DESC
LIMIT 5;
"""
cursor.execute(query3)
rows3 = cursor.fetchall()
for row in rows3:
    print(row)

print("\n" + "=" * 60)
print("Q4: Who are the 10 most active readers (most books borrowed)?")
print("=" * 60)
query4 = """
SELECT m.member_id, m.first_name, m.last_name, COUNT(c.checkout_id) AS total_checkouts
FROM members m
JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY total_checkouts DESC
LIMIT 10;
"""
cursor.execute(query4)
rows4 = cursor.fetchall()
for row in rows4:
    print(row)

print("\n" + "=" * 60)
print("Q5: Neighborhood activity further back in time (skip the 10 most recent)")
print("=" * 60)
# Neighborhood chosen: Maadi
# Change NEIGHBORHOOD below if you want a different one
NEIGHBORHOOD = "Maadi"
query5 = """
SELECT c.checkout_id, m.member_id, m.first_name, m.last_name, m.neighborhood,
       c.book_id, c.checkout_date
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE m.neighborhood = ?
ORDER BY c.checkout_date DESC
LIMIT -1 OFFSET 10;
"""
cursor.execute(query5, (NEIGHBORHOOD,))
rows5 = cursor.fetchall()
for row in rows5:
    print(row)
print(f"Neighborhood used: {NEIGHBORHOOD}")

conn.close()


Q1: How much is each member borrowing? (per-member checkout count, including zero)
(1034, 'Aya', 'Wahba', 25)
(1044, 'Sherif', 'Saleh', 21)
(1008, 'Ziad', 'Saleh', 19)
(1010, 'Nour', 'Nabil', 18)
(1027, 'Mostafa', 'Fouad', 18)
(1018, 'Ahmed', 'Shafik', 17)
(1024, 'Youssef', 'Hegazy', 17)
(1065, 'Adam', 'Fahmy', 17)
(1030, 'Reem', 'Osman', 16)
(1047, 'Sara', 'Rashad', 16)
(1072, 'Seif', 'Zaki', 14)
(1050, 'Fares', 'Adel', 11)
(1057, 'Ahmed', 'Fahmy', 10)
(1079, 'Rana', 'Osman', 10)
(1003, 'Bassel', 'Hegazy', 9)
(1075, 'Malak', 'Fahmy', 9)
(1061, 'Ziad', 'Fahmy', 8)
(1016, 'Dina', 'Rashad', 7)
(1020, 'Rana', 'Fouad', 7)
(1059, 'Lina', 'Halim', 7)
(1062, 'Tarek', 'Adel', 7)
(1076, 'Dina', 'Wahba', 7)
(1022, 'Youssef', 'Fouad', 6)
(1032, 'Nada', 'Zaki', 6)
(1054, 'Retaj', 'Fahmy', 6)
(1068, 'Layla', 'Hegazy', 6)
(1077, 'Lina', 'Rashad', 6)
(1053, 'Adam', 'Shafik', 5)
(1005, 'Youssef', 'Halim', 3)
(1007, 'Sherif', 'Fouad', 3)
(1015, 'Hamza', 'Sabry', 3)
(1019, 'Mostafa', 'Wahba', 3)
(1026, 

### Part 2 - Combining the Three Sources
Stage 1 (members + checkouts in Python), Stage 2 (add book details), Stage 3 (add Reading Kickoff checkouts).

In [2]:
"""
Task 1 - Part 2: Combine the three sources into one unified dataset.

Stage 1: Members + Checkouts, joined in plain Python (no SQL JOIN for this step).
Stage 2: Add book details (title/author from the database, plus genre/pages/
         publication_year/publisher from the Book Catalog JSON).
Stage 3: Add the Reading Kickoff checkouts from the HTML page, in the same shape.
"""

import sqlite3
import json
import re
import pandas as pd

DB_PATH = "Library Database"
CATALOG_PATH = "Book Catalog"
KICKOFF_PATH = "Reading Kickoff Signups"

# -----------------------------------------------------------------
# Load raw data from each source
# -----------------------------------------------------------------
conn = sqlite3.connect(DB_PATH)

# We only use SQL here to pull whole tables out (SELECT *), not to do the join.
# The actual joining logic below is done with plain Python / pandas.
members_df = pd.read_sql_query("SELECT * FROM members;", conn)
checkouts_df = pd.read_sql_query("SELECT * FROM checkouts;", conn)
books_df = pd.read_sql_query("SELECT * FROM books;", conn)

conn.close()

with open(CATALOG_PATH, "r", encoding="utf-8") as f:
    catalog_list = json.load(f)
catalog_df = pd.DataFrame(catalog_list)

# -----------------------------------------------------------------
# Stage 1: Members + Checkouts (done in Python, no SQL join)
# -----------------------------------------------------------------
# Build a lookup dictionary of member_id -> member info (plain Python, not SQL)
members_lookup = {}
for _, row in members_df.iterrows():
    members_lookup[row["member_id"]] = {
        "first_name": row["first_name"],
        "last_name": row["last_name"],
        "grade": row["grade"],
        "neighborhood": row["neighborhood"],
        "membership_status": row["membership_status"],
        "join_date": row["join_date"],
    }

stage1_rows = []
for _, row in checkouts_df.iterrows():
    member_info = members_lookup.get(row["member_id"], {})
    combined_row = {
        "checkout_id": row["checkout_id"],
        "member_id": row["member_id"],
        "first_name": member_info.get("first_name"),
        "last_name": member_info.get("last_name"),
        "grade": member_info.get("grade"),
        "neighborhood": member_info.get("neighborhood"),
        "membership_status": member_info.get("membership_status"),
        "book_id": row["book_id"],
        "checkout_date": row["checkout_date"],
        "return_date": row["return_date"],
        "source": "database",
    }
    stage1_rows.append(combined_row)

stage1_df = pd.DataFrame(stage1_rows)

# Sanity check: no checkout should be lost
assert len(stage1_df) == len(checkouts_df), "Stage 1: checkout count changed unexpectedly!"

# Per-member total checkout count (bonus column requested in the task)
member_totals = stage1_df.groupby("member_id")["checkout_id"].transform("count")
stage1_df["member_total_checkouts"] = member_totals

print(f"Stage 1 done. Rows: {len(stage1_df)}")

# -----------------------------------------------------------------
# Stage 2: Add book details (title/author from DB + catalog details)
# -----------------------------------------------------------------
# Merge title/author from the books table first
book_titles_authors = books_df[["book_id", "title", "author"]]
stage2_df = stage1_df.merge(book_titles_authors, on="book_id", how="left")

# Merge extra details from the catalog JSON
stage2_df = stage2_df.merge(catalog_df, on="book_id", how="left")

# Sanity check: number of checkouts must NOT change in this stage
assert len(stage2_df) == len(stage1_df), "Stage 2: row count changed unexpectedly!"

print(f"Stage 2 done. Rows: {len(stage2_df)}")

# -----------------------------------------------------------------
# Stage 3: Add the Reading Kickoff checkouts (from the HTML page)
# -----------------------------------------------------------------
with open(KICKOFF_PATH, "r", encoding="utf-8") as f:
    html_content = f.read()

# Extract every <tr>...</tr> row, skipping the header row
row_pattern = re.compile(r"<tr>(.*?)</tr>", re.DOTALL)
cell_pattern = re.compile(r"<td>(.*?)</td>", re.DOTALL)

kickoff_records = []
for row_match in row_pattern.findall(html_content):
    cells = cell_pattern.findall(row_match)
    if len(cells) == 3:  # skips the header row, which uses <th> not <td>
        member_id, book_id, checkout_date = cells
        kickoff_records.append({
            "member_id": int(member_id.strip()),
            "book_id": int(book_id.strip()),
            "checkout_date": checkout_date.strip(),
        })

kickoff_df = pd.DataFrame(kickoff_records)
print(f"Reading Kickoff rows found in HTML: {len(kickoff_df)}")

# Assign new checkout_ids that don't collide with existing ones
next_id = int(stage2_df["checkout_id"].max()) + 1
kickoff_df["checkout_id"] = range(next_id, next_id + len(kickoff_df))
kickoff_df["return_date"] = None  # Reading Kickoff checkouts have no return date on record
kickoff_df["source"] = "reading_kickoff"

# Attach member info the same way as Stage 1
def get_member_field(member_id, field):
    info = members_lookup.get(member_id, {})
    return info.get(field)

for field in ["first_name", "last_name", "grade", "neighborhood", "membership_status"]:
    kickoff_df[field] = kickoff_df["member_id"].apply(lambda m: get_member_field(m, field))

# Attach book details the same way as Stage 2
kickoff_df = kickoff_df.merge(book_titles_authors, on="book_id", how="left")
kickoff_df = kickoff_df.merge(catalog_df, on="book_id", how="left")

# Combine everything into the final dataset
final_df = pd.concat([stage2_df, kickoff_df], ignore_index=True)

# Recompute member_total_checkouts now that Kickoff checkouts are included
final_df["member_total_checkouts"] = final_df.groupby("member_id")["checkout_id"].transform("count")

print(f"Stage 3 done. Final combined rows: {len(final_df)}")
print(f"Final columns: {list(final_df.columns)}")
print("\nSample rows:")
print(final_df.sample(min(5, len(final_df))))

# -----------------------------------------------------------------
# Save the final combined dataset
# -----------------------------------------------------------------
OUTPUT_NAME = "combined_dataset.csv"  # Rename per the "Student ID-Library" naming rule before submitting
final_df.to_csv(OUTPUT_NAME, index=False)
print(f"\nSaved combined dataset to {OUTPUT_NAME}")


Stage 1 done. Rows: 391
Stage 2 done. Rows: 391
Reading Kickoff rows found in HTML: 26
Stage 3 done. Final combined rows: 417
Final columns: ['checkout_id', 'member_id', 'first_name', 'last_name', 'grade', 'neighborhood', 'membership_status', 'book_id', 'checkout_date', 'return_date', 'source', 'member_total_checkouts', 'title', 'author', 'genre', 'pages', 'publication_year', 'publisher']

Sample rows:
     checkout_id  member_id first_name last_name  grade neighborhood  \
49          9139       1036       Lina      Zaki    7.0    Nasr City   
242         9201       1055    Mostafa     Kamel    9.0   Heliopolis   
28          9152       1030       Reem     Osman    NaN    Nasr City   
97          9376       1024    Youssef    Hegazy    8.0    Nasr City   
385         9038       1007     Sherif     Fouad    6.0        Maadi   

    membership_status  book_id checkout_date return_date    source  \
49             Active      521    2024-03-14  2024-03-20  database   
242            Active

## Task 2: Data Integrity
Exploring and resolving the four data quality problems in the combined dataset.

In [3]:
"""
Task 2 - Step 1: Explore the four data quality problems before fixing anything.
This script does NOT modify the data - it only reports what's wrong, where,
and how much, so we can decide how to handle each issue.
"""

import pandas as pd
import sqlite3

INPUT_PATH = "EYOUTH-31003301403574-Library-task1_combined_data.csv"

df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns.\n")

# -----------------------------------------------------------------
# Problem 1: Missing values, column by column
# -----------------------------------------------------------------
print("=" * 60)
print("PROBLEM 1: Missing values per column")
print("=" * 60)
missing_counts = df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0]
print(missing_counts)
print()

# -----------------------------------------------------------------
# Problem 2: Duplicate records
# -----------------------------------------------------------------
print("=" * 60)
print("PROBLEM 2: Duplicate records")
print("=" * 60)
# True duplicates: every column matches exactly
exact_dupes = df[df.duplicated(keep=False)]
print(f"Exact full-row duplicates: {len(exact_dupes)} rows involved")
if len(exact_dupes) > 0:
    print(exact_dupes.sort_values(by=list(df.columns)).head(10))

# Also check duplicates based only on checkout_id (should be unique)
checkout_id_dupes = df[df.duplicated(subset=["checkout_id"], keep=False)]
print(f"\nRows sharing the same checkout_id: {len(checkout_id_dupes)}")
if len(checkout_id_dupes) > 0:
    print(checkout_id_dupes.sort_values(by="checkout_id").head(10))
print()

# -----------------------------------------------------------------
# Problem 3: Inconsistent text values
# -----------------------------------------------------------------
print("=" * 60)
print("PROBLEM 3: Inconsistent text values")
print("=" * 60)
print("Unique values in 'neighborhood':")
print(df["neighborhood"].dropna().unique())
print("\nUnique values in 'membership_status':")
print(df["membership_status"].dropna().unique())
print("\nUnique values in 'genre' (checking just in case):")
print(df["genre"].dropna().unique())
print()

# -----------------------------------------------------------------
# Problem 4: member_id values that don't exist among registered members
# -----------------------------------------------------------------
print("=" * 60)
print("PROBLEM 4: Checkouts referencing a non-existent member_id")
print("=" * 60)
conn = sqlite3.connect("Library Database")
real_members = pd.read_sql_query("SELECT member_id FROM members;", conn)
conn.close()

real_member_ids = set(real_members["member_id"])
invalid_rows = df[~df["member_id"].isin(real_member_ids)]
print(f"Rows with a member_id not found in the members table: {len(invalid_rows)}")
if len(invalid_rows) > 0:
    print(invalid_rows[["checkout_id", "member_id", "first_name", "last_name", "source"]])


Loaded 417 rows, 18 columns.

PROBLEM 1: Missing values per column
first_name            5
last_name             5
grade                41
neighborhood          5
membership_status     5
return_date          91
publication_year     35
dtype: int64

PROBLEM 2: Duplicate records
Exact full-row duplicates: 16 rows involved
     checkout_id  member_id first_name last_name  grade neighborhood  \
113         9052       1019    Mostafa     Wahba    6.0        Maadi   
357         9052       1019    Mostafa     Wahba    6.0        Maadi   
246         9180       1034        Aya     Wahba    9.0    Nasr City   
278         9180       1034        Aya     Wahba    9.0    Nasr City   
121         9193       1034        Aya     Wahba    9.0    Nasr City   
375         9193       1034        Aya     Wahba    9.0    Nasr City   
116         9194       1024    Youssef    Hegazy    8.0    Nasr City   
333         9194       1024    Youssef    Hegazy    8.0    Nasr City   
330         9246       1044   

In [4]:
"""
Task 2 - Step 2: Resolve the four data quality problems and save the cleaned dataset.
"""

import pandas as pd

INPUT_PATH = "EYOUTH-31003301403574-Library-task1_combined_data.csv"
OUTPUT_PATH = "EYOUTH-31003301403574-Library-task2_cleaned_data.csv"

df = pd.read_csv(INPUT_PATH)
print(f"Starting rows: {len(df)}")

# -----------------------------------------------------------------
# Problem 2: Remove true duplicate records (exact full-row matches)
# -----------------------------------------------------------------
before = len(df)
df = df.drop_duplicates(keep="first")
print(f"Removed {before - len(df)} true duplicate rows. Rows now: {len(df)}")

# -----------------------------------------------------------------
# Problem 4: Remove checkouts referencing a member_id that doesn't exist
# -----------------------------------------------------------------
import sqlite3
conn = sqlite3.connect("Library Database")
real_members = pd.read_sql_query("SELECT member_id FROM members;", conn)
conn.close()
real_member_ids = set(real_members["member_id"])

before = len(df)
df = df[df["member_id"].isin(real_member_ids)]
print(f"Removed {before - len(df)} rows with a non-existent member_id. Rows now: {len(df)}")

# -----------------------------------------------------------------
# Problem 3: Standardize inconsistent text values
# -----------------------------------------------------------------
# Neighborhood: strip whitespace and fix casing (Title Case)
df["neighborhood"] = df["neighborhood"].str.strip().str.title()

# Membership status: fix casing (Title Case) so Active/active become the same value
df["membership_status"] = df["membership_status"].str.strip().str.title()

print("\nNeighborhood values after cleanup:", sorted(df["neighborhood"].dropna().unique()))
print("Membership status values after cleanup:", sorted(df["membership_status"].dropna().unique()))

# -----------------------------------------------------------------
# Problem 1: Missing values, handled column by column
# -----------------------------------------------------------------
# grade: left as missing - no other source can tell us a member's real grade,
# so guessing a value would introduce false information.
#
# return_date: left as missing on purpose - a missing return_date means the
# book simply hasn't been returned yet (it's still checked out). This is
# meaningful information, not an error, so it must NOT be dropped or filled.
#
# publication_year: left as missing - the Book Catalog itself has gaps for
# some books, and there's no reliable way to recover the real year.
#
# first_name / last_name / neighborhood / membership_status: these were only
# missing for the 5 rows removed in Problem 4 (invalid member_id), so they
# should now show 0 missing values.

print("\nRemaining missing values per column after all fixes:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# -----------------------------------------------------------------
# Save the cleaned dataset
# -----------------------------------------------------------------
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved cleaned dataset to {OUTPUT_PATH}")
print(f"Final row count: {len(df)}")


Starting rows: 417
Removed 8 true duplicate rows. Rows now: 409
Removed 5 rows with a non-existent member_id. Rows now: 404

Neighborhood values after cleanup: ['Heliopolis', 'Maadi', 'Nasr City', 'Shubra', 'Zamalek']
Membership status values after cleanup: ['Active', 'Inactive']

Remaining missing values per column after all fixes:
grade               36
return_date         86
publication_year    33
dtype: int64

Saved cleaned dataset to EYOUTH-31003301403574-Library-task2_cleaned_data.csv
Final row count: 404


## Task 3: Data Fairness
Comparing member counts and checkout counts by neighborhood.

In [5]:
"""
Task 3 - Step 1: Compare neighborhoods by member count and checkout count.
This is exploratory - it prints the numbers we need to write the Fairness Reflection.
"""

import pandas as pd
import sqlite3

# Member counts per neighborhood come from the ORIGINAL members table
# (every registered member, regardless of whether they've ever checked out a book)
conn = sqlite3.connect("Library Database")
members_df = pd.read_sql_query("SELECT * FROM members;", conn)
conn.close()

# Normalize neighborhood text the same way Task 2 did, so comparisons are fair.
# Also collapse any internal double spaces (e.g. "Nasr  City" -> "Nasr City"),
# which is a separate small inconsistency found while building this comparison.
members_df["neighborhood"] = (
    members_df["neighborhood"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

member_counts = members_df.groupby("neighborhood")["member_id"].count()
print("Members per neighborhood:")
print(member_counts)

# Checkout counts come from the CLEANED dataset from Task 2
checkouts_df = pd.read_csv("EYOUTH-31003301403574-Library-task2_cleaned_data.csv")
checkout_counts = checkouts_df.groupby("neighborhood")["checkout_id"].count()
print("\nCheckouts per neighborhood:")
print(checkout_counts)

# Combine into one side-by-side table
comparison = pd.DataFrame({
    "member_count": member_counts,
    "checkout_count": checkout_counts,
})
comparison["checkouts_per_member"] = (comparison["checkout_count"] / comparison["member_count"]).round(2)
comparison = comparison.sort_values("checkouts_per_member")

print("\n" + "=" * 60)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 60)
print(comparison)


Members per neighborhood:
neighborhood
Heliopolis    18
Maadi         22
Nasr City     20
Shubra         6
Zamalek       14
Name: member_id, dtype: int64

Checkouts per neighborhood:
neighborhood
Heliopolis     87
Maadi         114
Nasr City     101
Shubra         34
Zamalek        68
Name: checkout_id, dtype: int64

SIDE-BY-SIDE COMPARISON
              member_count  checkout_count  checkouts_per_member
neighborhood                                                    
Heliopolis              18              87                  4.83
Zamalek                 14              68                  4.86
Nasr City               20             101                  5.05
Maadi                   22             114                  5.18
Shubra                   6              34                  5.67


## Conclusion
See `writeup_task1.txt`, `integrity_report.docx`, and `fairness_reflection.docx` for the full written analysis and reasoning behind each decision made in this notebook.